# Pre-process...



In [20]:
# Clone your GitHub repo
!git clone https://github.com/Brandenn28/ML.git

# Move into it
!cd ML-main/AML_project_herbarium_dataset

fatal: destination path 'ML' already exists and is not an empty directory.
/bin/bash: line 1: cd: ML-main/AML_project_herbarium_dataset: No such file or directory


In [21]:
!ls -R /content


/content:
 ML  'plant_classifier (1).pth'   plant_classifier_tinynet.pth	 sample_data

/content/ML:
AML_project_herbarium_dataset  DinoV2_Baseline_2.ipynb

/content/ML/AML_project_herbarium_dataset:
 list		       'Pre-Processing Summary.md'   test
 pre-processing.ipynb   processed		     train

/content/ML/AML_project_herbarium_dataset/list:
class_without_pairs.txt  groundtruth.txt   test.txt
class_with_pairs.txt	 species_list.txt  train.txt

/content/ML/AML_project_herbarium_dataset/processed:
class_counts.csv	    species_without_pairs.csv  train_sample_weights.csv
class_weights.csv	    species_with_pairs.csv     train_v2.csv
image_sizes.csv		    test.csv		       val.csv
per_class_domain_stats.csv  test_with_groundtruth.csv  val_v2.csv
species_mapping.csv	    train.csv

/content/ML/AML_project_herbarium_dataset/test:
1000.jpg    145749.jpg	196172.jpg  254491.jpg	265188.jpg  305719.jpg
1007.jpg    145752.jpg	197081.jpg  254506.jpg	265630.jpg  305896.jpg
1008.jpg    146208.jpg	197317.jpg

In [22]:
DATASET_ROOT = "/content/ML/AML_project_herbarium_dataset"
TRAIN_DIR = f"{DATASET_ROOT}/train"
TEST_DIR = f"{DATASET_ROOT}/test"
PROCESSED_DIR = f"{DATASET_ROOT}/processed"
LIST_DIR = f"{DATASET_ROOT}/list"

print("Successfully updated directory!")

Successfully updated directory!


In [23]:
import pandas as pd

train_df = pd.read_csv(f"{PROCESSED_DIR}/train.csv")
val_df = pd.read_csv(f"{PROCESSED_DIR}/val.csv")
test_df = pd.read_csv(f"{PROCESSED_DIR}/test_with_groundtruth.csv")

train_df.head()

,rel_path,abs_path,class_id,label_idx,species_name,domain
0,train/photo/106461/154340.jpg,D:\AML\AML dataset\Herbarium_Field dataset\AML...,106461,20,Costus scaber Ruiz Pav.,field
1,train/photo/106461/154115.jpg,D:\AML\AML dataset\Herbarium_Field dataset\AML...,106461,20,Costus scaber Ruiz Pav.,field
2,train/photo/106461/154345.jpg,D:\AML\AML dataset\Herbarium_Field dataset\AML...,106461,20,Costus scaber Ruiz Pav.,field
3,train/photo/106461/154119.jpg,D:\AML\AML dataset\Herbarium_Field dataset\AML...,106461,20,Costus scaber Ruiz Pav.,field
4,train/photo/106461/154344.jpg,D:\AML\AML dataset\Herbarium_Field dataset\AML...,106461,20,Costus scaber Ruiz Pav.,field


In [24]:
!pip install torch torchvision torchaudio --upgrade

In [25]:
import os
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim

# Custom dataset
class PlantDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None, label_map=None):
        self.data = dataframe
        self.img_dir = img_dir
        self.transform = transform
        self.label_map = label_map # Store label_map in the instance

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        rel_path = self.data.iloc[idx]['rel_path']
        img_path = os.path.join(DATASET_ROOT, rel_path)
        image = Image.open(img_path).convert("RGB")
        label = self.data.iloc[idx]['label']

        if self.transform:
            image = self.transform(image)
        return image, label

# Transforms
transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Preprocessing steps moved here, before dataset instantiation

# Build class mapping once from the training data
label_map = {name: idx for idx, name in enumerate(train_df['species_name'].unique())}

# Add 'label' column to dataframes using the created label_map
train_df['label'] = train_df['species_name'].map(label_map)
val_df['label'] = val_df['species_name'].map(label_map)
test_df['label'] = test_df['species_name'].map(label_map)

# Drop rows with NaN labels (species not found in train_df's unique species) if any
train_df = train_df.dropna(subset=['label'])
val_df = val_df.dropna(subset=['label'])
test_df = test_df.dropna(subset=['label'])

# Convert label column to integer type
train_df['label'] = train_df['label'].astype(int)
val_df['label'] = val_df['label'].astype(int)
test_df['label'] = test_df['label'].astype(int)

# Build weights
class_counts = train_df['label'].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_df['label']]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# Load datasets - now passing pre-processed dataframe and label_map
train_dataset = PlantDataset(train_df, TRAIN_DIR, transform, label_map)
test_dataset = PlantDataset(test_df, TEST_DIR, transform, label_map)

train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [26]:
!pip install focal-loss-torch

# Model Training

In [ ]:
from focal_loss.focal_loss import FocalLoss

# Model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model= models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
num_classes = len(train_dataset.label_map)
in_features = model.classifier[2].in_features # Get model features
model.classifier = nn.Sequential(
    nn.Flatten(),
    nn.Linear(in_features, 512), # Intermediate layer to undergo further dropouts
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, num_classes)  # Final linear layer to output num_classes
)
model = model.to(device);

criterion = FocalLoss(gamma=2)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth


100%|██████████| 338M/338M [00:04<00:00, 71.6MB/s]


In [ ]:
torch.cuda.empty_cache()

In [ ]:
import torch.nn.functional as F

for epoch in range(12):  # in the beginning it was 5 epochs, to fine tune further, this would be 12
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        probabilities = F.softmax(outputs, dim=1)
        loss = criterion(probabilities, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch [{epoch+1}/12], Loss: {total_loss/len(train_loader):.4f}")

Epoch [1/12], Loss: 3.8808
Epoch [2/12], Loss: 2.4980
Epoch [3/12], Loss: 1.6873
Epoch [4/12], Loss: 1.1790
Epoch [5/12], Loss: 0.9497
Epoch [6/12], Loss: 0.7358
Epoch [7/12], Loss: 0.5304
Epoch [8/12], Loss: 0.4620
Epoch [9/12], Loss: 0.3483
Epoch [10/12], Loss: 0.3277
Epoch [11/12], Loss: 0.2840
Epoch [12/12], Loss: 0.2401


In [ ]:
import torch
import numpy as np

def evaluate_model(model, data_loader, num_classes, device):
    """
    Evaluates a classification model using:
      - Top-1 Accuracy (overall accuracy)
      - Top-5 Accuracy
    """
    model.eval()
    top1_correct = 0
    top5_correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            # TOP-1
            _, top1_pred = torch.max(outputs, 1)
            top1_correct += (top1_pred == labels).sum().item()

            # TOP-5
            _, top5_pred = outputs.topk(5, dim=1)
            top5_correct += top5_pred.eq(labels.view(-1, 1)).sum().item()

            total += labels.size(0)

    top1_acc = top1_correct / total
    top5_acc = top5_correct / total

    return top1_acc, top5_acc

In [ ]:
top1_acc, top5_acc = evaluate_model(
    model, test_loader, num_classes, device
)
print("Top-1 Accuracy: ", round(top1_acc*100, 2), "%")
print("Top-5 Accuracy: ", round(top5_acc*100, 2), "%")

Top-1 Accuracy:  53.62 %
Average Accuracy per Class:  43.67 %


In [ ]:
torch.save(model.state_dict(), "plant_classifier.pth")

import json
with open("label_map.json", "w") as f:
    json.dump(train_dataset.label_map, f)

print("✅ Model and labels saved successfully.")

✅ Model and labels saved successfully.


# Evaluation

In [30]:
import torch
from torchvision import datasets, transforms, models

tiny_model_path = "/content/plant_classifier_tinynet.pth"
base_model_path = "/content/plant_classifier (1).pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.convnext_tiny(num_classes=100)
state_dict = torch.load(tiny_model_path, map_location=torch.device('cpu'))
model.load_state_dict(state_dict)
model.eval()
model = model.to(device);

In [28]:
!pip install torchmetrics

In [31]:
import torch
from torchmetrics.classification import MulticlassAccuracy

num_classes = 100

top1 = MulticlassAccuracy(num_classes=num_classes, top_k=1)
top5 = MulticlassAccuracy(num_classes=num_classes, top_k=5)

top1_acc = 0.0
top5_acc = 0.0
count = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        top1_acc += top1(outputs, labels).item()
        top5_acc += top5(outputs, labels).item()
        count += 1

top1_acc /= count
top5_acc /= count

print(f"Top-1 Accuracy: {top1_acc:.4f}")
print(f"Top-5 Accuracy: {top5_acc:.4f}")

Top-1 Accuracy: 0.1332
Top-5 Accuracy: 0.4845


In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Re-initialize the model with its base structure, without pre-trained classifier weights
model = models.convnext_base(weights=None)

# Get the in_features for the new custom classifier. For convnext_base,
# the default classifier is Sequential(LayerNorm, Flatten, Linear), so we get it from the Linear layer.
in_features = model.classifier[2].in_features # This should be 1024
num_classes = len(train_dataset.label_map) # Ensure num_classes is consistent

# Reconstruct the custom classifier exactly as it was defined before saving
model.classifier = nn.Sequential(
    nn.Flatten(),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, num_classes)
)

# Now load the state_dict into the model with the correct architecture
state_dict = torch.load(base_model_path, map_location=torch.device('cpu'))
model.load_state_dict(state_dict)
model.eval()
model = model.to(device);

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Re-initialize the model with its base structure, without pre-trained classifier weights
model = models.convnext_base(weights=None)

# Get the in_features for the new custom classifier. For convnext_base,
# the default classifier is Sequential(LayerNorm, Flatten, Linear), so we get it from the Linear layer.
in_features = model.classifier[2].in_features # This should be 1024
num_classes = len(train_dataset.label_map) # Ensure num_classes is consistent

# Reconstruct the custom classifier exactly as it was defined before saving
model.classifier = nn.Sequential(
    nn.Flatten(),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, num_classes)
)

# Now load the state_dict into the model with the correct architecture
state_dict = torch.load(base_model_path, map_location=torch.device('cpu'))
model.load_state_dict(state_dict)
model.eval()
model = model.to(device);

In [40]:
import torch
from torchmetrics.classification import MulticlassAccuracy

num_classes = 100

top1 = MulticlassAccuracy(num_classes=num_classes, top_k=1)
top5 = MulticlassAccuracy(num_classes=num_classes, top_k=5)

top1_acc = 0.0
top5_acc = 0.0
count = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        top1_acc += top1(outputs, labels).item()
        top5_acc += top5(outputs, labels).item()
        count += 1

top1_acc /= count
top5_acc /= count

print(f"Top-1 Accuracy: {top1_acc:.4f}")
print(f"Top-5 Accuracy: {top5_acc:.4f}")

Top-1 Accuracy: 0.3359
Top-5 Accuracy: 0.6408
